这一章是 Go 后端面试里非常能拉开差距的一章。

很多人学 Go 只停留在语法层面，能写 goroutine、channel、slice、map，但一问到 runtime，就会回答得很虚。实际上，对于后端工程来说，Go runtime 至少要掌握五条线：

第一，Go 对象到底分配在栈上还是堆上。
Java 里常说 new 出来的对象通常在堆上，局部变量在栈上。但 Go 不能这么机械理解。Go 里变量最终在栈还是堆，核心取决于逃逸分析，而不是你有没有写 new，也不是你有没有取地址。

第二，Go 内存分配器是怎么工作的。
Go 的内存分配不是每次都直接向操作系统申请，而是 runtime 自己维护一套内存管理体系，包括 mspan、mcache、mcentral、mheap、size class、tiny allocator 等。理解这条线，才能解释为什么 Go 小对象分配很快，也能理解为什么大量小对象会造成 GC 压力。

第三，Go GC 是怎么工作的。
Go 使用并发三色标记清扫 GC。它不是完全没有 STW，而是尽量把 STW 控制得很短。核心概念包括 GC Roots、三色标记、写屏障、STW、GC pacer、GOGC、GC assist。

第四，Go 性能调优怎么做。
Go 调优不是背参数，而是先看现象，再用工具定位。常见现象包括 CPU 高、内存涨、GC 频繁、goroutine 泄漏、锁竞争严重、阻塞严重、接口 RT 抖动。

第五，pprof 怎么用。
pprof 是 Go 工程里最重要的性能分析工具之一。CPU profile、heap profile、goroutine profile、mutex profile、block profile 都要知道看什么、怎么判断、如何优化。

本章最终要形成一句话：

Go runtime 面试的主线是：变量通过逃逸分析决定栈堆位置，小对象通过 runtime 分配器高效分配，GC 通过并发三色标记控制延迟，线上调优通过 pprof 和 runtime 指标定位 CPU、内存、协程、锁和阻塞问题。

---

# 一、Go Runtime 是什么

## 【文本块 2】Q：Go runtime 主要负责什么？

Go runtime 可以理解为 Go 程序运行时的一层基础设施。

Go 是编译型语言，代码会被编译成本地机器码。但 Go 并不是完全“裸跑”在操作系统之上，它仍然依赖 runtime 提供很多能力。

runtime 主要负责：

1. goroutine 调度
   也就是 GMP 调度模型，让大量 goroutine 复用少量操作系统线程执行。

2. 内存分配
   Go 程序里的对象创建、小对象分配、大对象分配，都由 runtime 内存分配器管理。

3. 垃圾回收
   Go 有自动 GC，runtime 负责发现不再使用的对象并回收内存。

4. 栈管理
   goroutine 的栈不是固定很大，而是可以按需增长和收缩，这也由 runtime 管理。

5. map、channel、interface、defer、panic/recover 等语言特性的底层支持
   这些看起来是语法或内置类型，但背后都有 runtime 参与。

6. 网络轮询
   Go 网络 IO 能用同步写法支撑高并发，底层依赖 runtime netpoller。

7. 反射和类型元信息
   reflect、interface 动态类型判断等，也和 runtime 类型信息相关。

面试里可以这样回答：

Go runtime 是 Go 程序运行时的基础设施，主要负责 goroutine 调度、内存分配、垃圾回收、栈管理、网络轮询，以及 map、channel、defer、panic 等语言特性的底层支持。虽然 Go 是编译型语言，但它仍然有比较重的 runtime，这也是 Go 能提供自动 GC 和轻量并发的重要原因。

---

## 【文本块 3】Q：Go 和 Java 都有运行时，它们有什么区别？

Java 代码运行在 JVM 上，源码先编译成字节码，运行时由 JVM 解释执行或 JIT 编译成本地机器码。JVM 是一个相对独立的虚拟机层，负责类加载、字节码执行、JIT、GC、线程管理等。

Go 则是 AOT 编译，通常直接编译成本地机器码。Go 没有 JVM 那种字节码虚拟机，也没有传统意义上的运行时 JIT。Go runtime 会被链接进最终二进制文件，和业务代码一起运行。

所以两者区别可以这样说：

Java 的运行时核心是 JVM，程序依赖 JVM 执行字节码。
Go 的 runtime 是编译进二进制里的运行时库，负责调度、GC、内存、栈等能力。
Java 有类加载、字节码解释、JIT 优化；Go 更偏静态编译和轻量 runtime 支撑。

面试里不要简单说“Go 没有 runtime”。准确说是：

> Go 没有 JVM 这种虚拟机运行时，但 Go 有自己的 runtime，并且 runtime 在 goroutine 调度、GC、内存管理上非常核心。

---

# 二、栈和堆

## 【文本块 4】Q：Go 里的栈和堆怎么理解？

Go 程序运行时，内存大体也可以分成栈和堆。

栈主要用于函数调用过程中的局部变量、参数、返回值、调用信息等。
堆主要用于存放生命周期不容易在编译期确定、需要被多个地方共享、或者逃逸出当前函数作用域的对象。

但是 Go 和很多语言的不同点在于：

Go 的 goroutine 栈初始很小，可以动态增长。
Go 变量最终分配在栈还是堆，不是由 var、new、取地址这些语法简单决定，而是由编译器逃逸分析决定。

例如：

```go
func f() *int {
    x := 10
    return &x
}
```

在 C/C++ 里返回局部变量地址是危险的，因为函数结束后栈帧释放了。

但 Go 里这是安全的。编译器会发现 x 的地址逃出了函数作用域，于是把 x 分配到堆上。

所以 Go 面试里一定要记住：

> Go 中变量分配在栈还是堆，核心取决于逃逸分析。返回局部变量指针在 Go 中是安全的，因为编译器会让它逃逸到堆上。

---

## 【代码块 1】返回局部变量指针在 Go 中是安全的

```go
import "fmt"

func newInt() *int {
    x := 10
    return &x
}

p := newInt()
fmt.Println(*p)
```

---

## 【文本块 5】代码解释

这个代码在 Go 中是安全的。

虽然 x 是函数内部局部变量，但它的地址被返回给外部使用。编译器通过逃逸分析发现 x 不能放在函数栈帧里，否则函数返回后就失效了。

所以 x 会被分配到堆上，由 GC 管理生命周期。

面试里可以顺着说：

> Go 不是通过语法判断对象一定在栈还是堆，而是通过逃逸分析判断变量生命周期。如果变量可能在函数返回后继续被引用，就会逃逸到堆上。

---

# 三、逃逸分析

## 【文本块 6】Q：什么是逃逸分析？

逃逸分析是 Go 编译器在编译阶段做的一种静态分析。

它要回答的问题是：

> 一个变量能不能安全地分配在当前 goroutine 的栈上？

如果编译器能确定变量只在当前函数或当前调用链内部使用，生命周期很短，就可以分配在栈上。

如果变量的引用逃出了当前函数，比如被返回、被闭包捕获、被存到堆对象里、被接口动态调用导致编译器无法确定具体类型，就可能分配到堆上。

栈分配的好处：

* 分配速度快
* 函数返回自动释放
* 不需要 GC 扫描和回收
* 对内存局部性更友好

堆分配的代价：

* 分配成本更高
* 增加 GC 扫描压力
* 对象生命周期更复杂
* 大量堆对象会导致 GC 频繁

所以性能优化里经常会说：

> 减少不必要的逃逸，本质上是减少堆分配和 GC 压力。

---

## 【文本块 7】常见逃逸场景

常见导致变量逃逸的场景有：

第一，返回局部变量指针。

```go
func f() *int {
    x := 1
    return &x
}
```

x 会逃逸到堆上。

第二，局部变量被闭包捕获，并且闭包生命周期可能超过当前函数。

```go
func f() func() int {
    x := 1
    return func() int {
        return x
    }
}
```

x 可能逃逸。

第三，把局部变量的指针存到全局变量或堆对象里。

```go
var gp *int

func f() {
    x := 1
    gp = &x
}
```

x 会逃逸。

第四，传给 interface{} 导致编译器无法完全静态确定使用方式。

```go
fmt.Println(x)
```

某些情况下可能导致逃逸，尤其是复杂对象传入接口。

第五，slice、map 中存储指针或大对象引用，导致对象生命周期延长。

第六，过大的对象可能直接分配到堆上。
即使没有明显逃逸，如果对象太大，放在栈上不合适，也可能分配到堆上。

面试里可以这样回答：

逃逸分析不是只看有没有取地址，而是看变量的生命周期是否超出了当前函数栈帧。如果变量可能在函数返回后继续被访问，或者编译器无法证明它只在栈内使用，就会逃逸到堆上。

---

## 【代码块 2】用 go build 查看逃逸分析

```go
// 保存为 escape.go 后，在终端执行：
// go build -gcflags="-m" escape.go

package main

func createPointer() *int {
    x := 10
    return &x
}

func main() {
    _ = createPointer()
}
```

---

## 【文本块 8】notebook 中怎么理解这段代码？

这段代码在 notebook 里可以阅读，但逃逸分析输出需要在终端里执行：

```bash
go build -gcflags="-m" escape.go
```

常见输出会类似：

```text
moved to heap: x
```

这说明 x 被移动到了堆上。

如果想看更详细的信息，可以用：

```bash
go build -gcflags="-m -m" escape.go
```

面试中不需要背具体输出格式，但要知道 Go 可以通过 `-gcflags="-m"` 查看逃逸分析结果。

---

## 【代码块 3】闭包捕获导致变量可能逃逸

```go
import "fmt"

func counter() func() int {
    x := 0

    return func() int {
        x++
        return x
    }
}

c := counter()

fmt.Println(c())
fmt.Println(c())
fmt.Println(c())
```

---

## 【文本块 9】代码解释

counter 函数返回了一个闭包。闭包内部继续使用 x。

虽然 counter 已经返回，但 x 仍然需要存在，因为闭包还要访问它。

所以 x 的生命周期超过了 counter 的栈帧，编译器通常会让它逃逸到堆上。

---

## 【文本块 10】Q：new 出来的对象一定在堆上吗？

不一定。

这是 Go 面试里很经典的坑。

很多人会从 Java 思维出发，说 new 出来的对象一定在堆上。Go 里不能这么说。

Go 的 new 只是分配一个零值对象并返回指针。至于这个对象最终在栈上还是堆上，仍然取决于逃逸分析。

例如：

```go
func f() {
    p := new(int)
    *p = 10
}
```

如果 p 没有逃出函数，编译器可能把这个 int 分配在栈上，甚至进一步优化掉。

所以面试答案是：

> Go 中 new 不等于堆分配。new 只是语义上创建零值对象并返回指针，最终位置由逃逸分析决定。

---

## 【代码块 4】new 不一定导致堆分配

```go
// 保存为 new_escape.go 后，在终端执行：
// go build -gcflags="-m" new_escape.go

package main

func f() {
    p := new(int)
    *p = 10
}

func main() {
    f()
}
```

---

## 【文本块 11】Q：如何减少不必要的逃逸？

常见优化思路有：

第一，避免不必要地返回指针。
小对象、不可变对象、简单值可以直接返回值。

第二，避免把短生命周期对象放进 interface{}。
尤其是热点路径里，不要为了通用而滥用 interface{} 和反射。

第三，减少闭包捕获大对象。
闭包很方便，但如果捕获了大对象或指针，可能延长生命周期。

第四，复用 buffer。
比如使用 bytes.Buffer、strings.Builder、sync.Pool，减少临时对象分配。

第五，避免在循环中频繁创建临时对象。
热点循环里大量小对象分配会增加 GC 压力。

第六，写 benchmark，并用 `-benchmem` 看分配次数。
不要凭感觉优化，要看 allocs/op 和 B/op。

面试里可以这样回答：

减少逃逸的核心是减少堆分配。具体做法包括：小对象能传值就传值，避免不必要的指针返回，避免热点路径 interface{} 和反射，减少闭包捕获，复用 buffer，并用 benchmark 的 allocs/op 验证优化效果。

---

# 四、Go 内存分配器总览

## 【文本块 12】Q：Go 内存分配器是怎么设计的？

Go 的内存分配器借鉴了 TCMalloc 的思想，核心目标是高效分配小对象，并减少多线程分配时的锁竞争。

Go 不是每次分配对象都直接向操作系统申请内存。
runtime 会先从操作系统申请较大的内存块，再在用户态自己切分和管理。

主要结构包括：

1. mheap
   全局堆管理器，负责向操作系统申请和归还大块内存。

2. mcentral
   按 size class 管理 span，每种规格的对象都有对应的 central。

3. mcache
   每个 P 都有自己的本地缓存，用于无锁或少锁地分配小对象。

4. mspan
   一段连续内存页，用来切分成相同大小的对象槽位。

5. size class
   Go 会把小对象按大小归类到不同规格，比如 8B、16B、32B、64B 等，不是每个对象大小都单独管理。

分配小对象的大致流程是：

1. 当前 goroutine 在某个 P 上运行。
2. 分配对象时，优先从当前 P 的 mcache 找对应 size class 的 mspan。
3. 如果 mcache 里有空闲槽位，直接分配，速度很快，基本不需要加锁。
4. 如果 mcache 没有，就去 mcentral 获取新的 mspan。
5. 如果 mcentral 也没有，就向 mheap 申请。
6. mheap 不够时，再向操作系统申请内存。

面试里可以这样回答：

Go 内存分配器采用分级缓存设计。小对象优先从当前 P 的 mcache 分配，避免频繁加锁；mcache 不够再找 mcentral，mcentral 不够再找全局 mheap，最后才向操作系统申请。通过 mspan 和 size class，Go 可以高效管理大量小对象。

---

## 【文本块 13】内存分配结构速记

可以按这条线记：

```text
goroutine
   ↓
P 的 mcache
   ↓
mcentral
   ↓
mheap
   ↓
操作系统
```

mcache：每个 P 本地缓存，分配最快。
mcentral：全局按规格管理 span，需要加锁。
mheap：全局堆，负责大块内存和向 OS 申请。
mspan：连续页组成的内存块，切成固定大小对象。
size class：对象大小规格分类，减少碎片和管理复杂度。

一句话：

> 小对象先走 mcache，大对象和 mcache 不够时才逐层向上申请。

---

# 五、小对象、大对象、tiny allocator

## 【文本块 14】Q：Go 小对象和大对象分配有什么区别？

Go runtime 会根据对象大小选择不同分配路径。

小对象会按 size class 分配。
runtime 从 mcache 中找到对应规格的 mspan，然后从空闲槽位中取一个给对象使用。

大对象不会塞进小对象 size class，而是直接从 mheap 分配一段足够大的连续页。

小对象分配关注的是速度和减少锁竞争。
大对象分配关注的是连续空间、页管理和 GC 成本。

此外，Go 还有 tiny allocator，用来优化非常小且不含指针的对象分配。

tiny allocator 会把多个很小的无指针对象合并到一个小块里，减少分配次数和内存浪费。

例如很多临时的小字符串头、小结构体、无指针小对象，都可能受益于 tiny allocator。

面试里可以这样回答：

Go 小对象通常通过 size class 从 mcache 的 mspan 中分配，大对象直接走 mheap。对于非常小且不含指针的对象，runtime 还有 tiny allocator，把多个小对象合并分配，进一步降低分配成本和内存碎片。

---

## 【文本块 15】Q：为什么 Go 小对象分配快？

主要有三个原因：

第一，mcache 是每个 P 本地的。
goroutine 在 P 上运行时，可以优先从本地 mcache 分配小对象，减少全局锁竞争。

第二，size class 让分配规格标准化。
runtime 不需要为每个对象维护任意大小的内存块，而是按规格快速找到合适槽位。

第三，mspan 已经提前切分好对象槽。
分配时很多情况下只是从空闲列表或 bitmap 中找到一个空位，不需要每次都向 OS 申请。

所以小对象分配路径很短。

但是这不代表可以无节制创建小对象。大量小对象仍然会增加 GC 扫描和回收压力。

---

# 六、对象分配和 GC 压力

## 【文本块 16】Q：为什么大量小对象会导致性能问题？

很多人会说 Go 分配很快，所以多创建点对象没关系。这个理解不完整。

Go 小对象分配确实比较快，但对象只要进入堆，就会带来 GC 成本。

GC 需要扫描对象图，找出哪些对象还活着。
如果堆上有大量对象，尤其是大量带指针对象，GC 标记阶段压力会变大。

大量小对象的问题主要有：

1. 分配次数多，allocs/op 高
2. 堆对象多，GC 扫描压力大
3. GC 更频繁，CPU 被 GC 消耗
4. 内存局部性差，缓存命中率下降
5. 临时对象很多，造成短生命周期垃圾

所以 Go 性能优化经常关注两个指标：

* B/op：每次操作分配多少字节
* allocs/op：每次操作发生多少次分配

面试里可以这样回答：

Go 分配小对象很快，但不代表堆分配没有成本。大量小对象会提高 allocs/op，增加堆对象数量和 GC 扫描压力，最终表现为 GC 频繁、CPU 消耗上升、接口延迟抖动。因此热点路径要尽量减少临时对象和不必要的堆分配。

---

## 【代码块 5】用 benchmark 观察分配

```go
// 保存为 alloc_test.go 后执行：
// go test -bench=. -benchmem

package main

import (
    "strconv"
    "testing"
)

func buildWithPlus(n int) string {
    s := ""
    for i := 0; i < n; i++ {
        s += strconv.Itoa(i)
    }
    return s
}

func BenchmarkBuildWithPlus(b *testing.B) {
    for i := 0; i < b.N; i++ {
        _ = buildWithPlus(100)
    }
}
```

---

## 【文本块 17】benchmem 看什么？

执行：

```bash
go test -bench=. -benchmem
```

重点看三列：

```text
ns/op
B/op
allocs/op
```

ns/op 表示每次操作耗时。
B/op 表示每次操作分配多少字节。
allocs/op 表示每次操作发生多少次内存分配。

如果优化前后 allocs/op 明显下降，通常意味着减少了堆分配，GC 压力也会下降。

---

## 【代码块 6】用 strings.Builder 减少字符串拼接分配

```go
// 保存为 alloc_test.go 后执行：
// go test -bench=. -benchmem

package main

import (
    "strconv"
    "strings"
    "testing"
)

func buildWithBuilder(n int) string {
    var b strings.Builder

    for i := 0; i < n; i++ {
        b.WriteString(strconv.Itoa(i))
    }

    return b.String()
}

func BenchmarkBuildWithBuilder(b *testing.B) {
    for i := 0; i < b.N; i++ {
        _ = buildWithBuilder(100)
    }
}
```

---

# 七、sync.Pool

## 【文本块 18】Q：sync.Pool 是什么？适合什么场景？

sync.Pool 是 Go 标准库提供的临时对象池。

它的主要作用是复用临时对象，减少频繁分配和 GC 压力。

典型使用场景：

* bytes.Buffer 复用
* 临时大对象复用
* 编解码 buffer
* 日志 buffer
* JSON 序列化临时对象
* 高并发请求中的短生命周期对象

sync.Pool 的特点：

第一，它是并发安全的。
多个 goroutine 可以同时 Get 和 Put。

第二，它适合缓存临时对象，不适合管理长期资源。
比如不能拿它当数据库连接池、TCP 连接池。

第三，Pool 里的对象可能随时被 GC 清掉。
所以不能依赖放进去的对象一定能取出来。

第四，Get 出来的对象状态要重置。
比如 bytes.Buffer 取出后要 Reset，避免上次使用残留数据。

面试里可以这样回答：

sync.Pool 用于复用临时对象，降低分配次数和 GC 压力，适合高并发下频繁创建的短生命周期对象。它不是通用资源池，Pool 中对象可能被 GC 清理，不能用于管理必须长期保持的连接或状态。

---

## 【代码块 7】sync.Pool 复用 bytes.Buffer

```go
import (
    "bytes"
    "fmt"
    "sync"
)

var bufferPool = sync.Pool{
    New: func() any {
        return new(bytes.Buffer)
    },
}

func buildMessage(name string) string {
    buf := bufferPool.Get().(*bytes.Buffer)
    defer bufferPool.Put(buf)

    buf.Reset()

    buf.WriteString("hello, ")
    buf.WriteString(name)

    return buf.String()
}

fmt.Println(buildMessage("go"))
fmt.Println(buildMessage("runtime"))
```

---

## 【文本块 19】sync.Pool 使用注意事项

第一，Put 回去前要确认对象可以复用。
如果对象里有用户数据、敏感数据、大引用，要清理干净。

第二，Get 出来后要重置状态。
比如 buffer 要 Reset，slice 要截断，struct 字段要清零。

第三，不要把 sync.Pool 当连接池。
连接池需要生命周期管理、健康检查、最大连接数控制，而 sync.Pool 没有这些语义。

第四，不要为了小收益滥用。
sync.Pool 适合热点路径中分配成本明显的对象。如果对象很小、分配不频繁，使用 sync.Pool 反而增加复杂度。

---

# 八、Go GC 总览

## 【文本块 20】Q：Go GC 是怎么工作的？

Go 使用自动垃圾回收，主要目标是低延迟。

现代 Go GC 的核心可以概括为：

> 并发三色标记清扫 GC。

大致流程包括：

1. STW，开启写屏障
   GC 开始时，需要短暂停顿所有 goroutine，做一些准备工作，比如开启写屏障。

2. 并发标记
   GC 线程和用户 goroutine 同时运行。GC 从根对象开始扫描，标记所有可达对象。

3. 标记终止
   再次短暂 STW，完成标记收尾，关闭写屏障。

4. 并发清扫
   回收未标记对象占用的内存，供后续分配复用。

Go GC 的重点不是追求最高吞吐，而是尽量把停顿时间控制得很短，适合在线服务。

面试里可以这样回答：

Go GC 是并发三色标记清扫 GC。它会从 GC Roots 出发标记可达对象，通过写屏障保证并发标记期间引用变化不会导致漏标。GC 过程中仍然有 STW，但主要 STW 阶段很短，大部分标记和清扫工作与用户 goroutine 并发执行。

---

## 【文本块 21】Q：什么是 GC Roots？

GC Roots 是垃圾回收时可达性分析的起点。

Go GC 会从这些根对象出发，沿着指针引用关系扫描所有可达对象。

常见 GC Roots 包括：

* goroutine 栈上的局部变量和参数
* 全局变量
* runtime 内部数据结构
* finalizer 相关对象
* 当前寄存器中的指针
* 一些特殊的系统栈和调度器结构

从 GC Roots 能访问到的对象就是存活对象。
访问不到的对象就是垃圾，可以被回收。

面试可以类比 Java：

Java 里也有 GC Roots，比如虚拟机栈、本地方法栈、静态变量、常量等。Go 也类似，只是具体运行时结构不同。

---

# 九、三色标记法

## 【文本块 22】Q：什么是三色标记法？

三色标记法是一种描述并发标记过程的抽象模型。

它把对象分成三种颜色：

白色：还没有被标记到。
灰色：已经被标记到了，但它引用的对象还没有全部扫描。
黑色：已经被标记到了，并且它引用的对象也扫描完成。

GC 开始时，所有对象都是白色。

然后从 GC Roots 出发，把根对象直接引用的对象标成灰色。

GC 不断从灰色集合中取对象，扫描它引用的其他对象：

* 被它引用的白色对象变成灰色
* 当前对象扫描完成后变成黑色

当没有灰色对象时，标记结束。
剩下的白色对象就是不可达对象，可以回收。

---

## 【文本块 23】三色标记为什么会有并发问题？

如果 GC 标记过程中，用户 goroutine 也在同时修改对象引用，就可能出现漏标。

经典漏标场景需要同时满足两个条件：

第一，黑色对象新增了对白色对象的引用。
第二，灰色对象删除了到这个白色对象的引用。

这样 GC 可能再也没有机会从灰色对象扫描到这个白色对象，但它其实已经被黑色对象引用，应该存活。结果它仍然保持白色，最后被错误回收。

这就是并发标记里的严重问题：把活对象当垃圾回收。

为了解决这个问题，GC 需要写屏障。

面试里可以这样回答：

三色标记在并发场景下的问题是用户线程可能同时修改引用关系，导致本来可达的白色对象被漏标。为避免这种情况，Go GC 使用写屏障，在引用写入时维护三色不变性，保证不会把活对象错误回收。

---

# 十、写屏障

## 【文本块 24】Q：写屏障是什么？

写屏障可以理解为 GC 在引用写入操作旁边插入的一段额外逻辑。

当程序修改对象指针字段时，写屏障会通知 GC：

> 引用关系发生变化了，你需要处理这个对象，避免并发标记漏掉它。

例如：

```go
obj.field = newObj
```

在 GC 并发标记开启期间，这种指针写入会触发写屏障逻辑。

写屏障不是一直开启。
通常在 GC 标记阶段开启，标记结束后关闭。否则每次指针写入都有额外成本。

面试里可以这样说：

写屏障是 GC 在并发标记期间维护正确性的机制。它在指针写入时执行额外逻辑，保证引用关系变化不会导致可达对象漏标。代价是写指针操作在 GC 期间会变慢一些。

---

## 【文本块 25】Q：Go GC 会不会 STW？

会。

Go GC 不是完全没有 STW。

只是 Go 的 GC 设计目标是把 STW 时间尽量控制得很短。

典型 STW 发生在：

1. GC 开始阶段
   需要暂停 goroutine，开启写屏障，做标记准备。

2. 标记终止阶段
   需要暂停 goroutine，完成标记收尾，关闭写屏障。

大部分标记工作和清扫工作是并发进行的。

所以准确回答是：

> Go GC 有 STW，但主要 STW 阶段很短，不是像传统 stop-the-world GC 那样长时间暂停整个程序。Go 通过并发标记、写屏障和 GC pacer 来控制延迟。

---

# 十一、GOGC 和 GC 触发

## 【文本块 26】Q：Go 什么时候触发 GC？

Go GC 常见触发方式有三类：

第一，堆内存增长达到阈值。
这是最主要的触发方式。

Go 会根据上次 GC 后的存活堆大小和 GOGC 计算下一次 GC 触发目标。

第二，手动调用 runtime.GC。
这会强制触发 GC，但线上业务代码一般不建议频繁手动调用。

第三，定时触发。
即使堆增长不明显，runtime 也可能在一定时间后触发 GC，避免长时间不回收。

其中最关键的是 GOGC。

GOGC 默认值通常是 100，含义可以粗略理解为：

> 下一次 GC 触发时，堆大小目标约等于上次 GC 后存活堆大小的 1 + GOGC/100。

如果 GOGC = 100，上次 GC 后存活对象是 100MB，那么下一次 GC 目标大约是堆增长到 200MB 左右触发。

如果 GOGC 调大，比如 200，GC 频率会降低，但内存占用会更高。
如果 GOGC 调小，比如 50，GC 更频繁，但内存占用更低。

面试里可以这样回答：

GOGC 控制的是 GC 触发的堆增长比例。GOGC 越大，允许堆增长越多，GC 频率越低，但内存占用更高；GOGC 越小，GC 更频繁，内存占用更低，但 CPU 消耗可能增加。

---

## 【代码块 8】查看和设置 GOGC

```go
import (
    "fmt"
    "runtime/debug"
)

old := debug.SetGCPercent(100)
fmt.Println("old GOGC:", old)

debug.SetGCPercent(old)
```

---

## 【文本块 27】线上应该怎么调 GOGC？

GOGC 不是越大越好，也不是越小越好。

如果服务内存很紧张，希望更积极回收，可以适当调低 GOGC。
代价是 GC 更频繁，CPU 消耗可能增加。

如果服务内存充足，希望减少 GC 频率、提高吞吐，可以适当调高 GOGC。
代价是堆会长得更大，RSS 更高。

调 GOGC 前要先看：

* 当前堆大小
* GC 频率
* GC CPU 占比
* p99 延迟是否受 GC 影响
* 机器内存余量
* 容器 memory limit
* 是否存在内存泄漏或对象分配异常

面试里可以这样回答：

GOGC 调优要结合内存余量和延迟目标。如果 GC 过于频繁且内存充足，可以适当调大 GOGC；如果内存紧张，可以调小 GOGC。但如果根因是对象分配过多或泄漏，单纯调 GOGC 只是缓解，不是根治。

---

# 十二、GC Pacer 和 GC Assist

## 【文本块 28】Q：什么是 GC pacer？

GC pacer 可以理解成 Go GC 的节奏控制器。

它的目标是控制 GC 工作量和程序分配速度之间的关系，避免两种极端：

第一，GC 干得太慢。
程序分配太快，堆超过目标太多，内存暴涨。

第二，GC 干得太猛。
GC 占用太多 CPU，影响业务 goroutine 执行。

GC pacer 会根据当前堆大小、分配速度、目标堆大小、标记进度等信息，动态调整 GC 工作节奏。

面试里不用讲太细源码，但要知道：

> GC pacer 的作用是让 GC 标记进度跟上内存分配速度，在内存占用和 CPU 消耗之间做平衡。

---

## 【文本块 29】Q：什么是 GC assist？

GC assist 可以理解成“谁分配得多，谁帮 GC 多干活”。

在 GC 标记阶段，如果用户 goroutine 分配内存太快，GC 后台标记跟不上，runtime 会让进行分配的 goroutine 协助做一部分标记工作。

这样可以避免堆增长失控。

但代价是：业务 goroutine 会被迫花时间帮 GC 干活，导致请求延迟上升。

所以如果线上看到 GC assist 时间明显，往往说明对象分配压力较大。

面试里可以这样回答：

GC assist 是 Go GC 的一种反压机制。GC 标记期间，如果业务 goroutine 分配太快，runtime 会让它协助完成一部分标记工作，保证 GC 进度跟上分配速度。它能控制内存增长，但会增加业务请求延迟。

---

# 十三、Go 内存泄漏

## 【文本块 30】Q：Go 有 GC，还会内存泄漏吗？

会。

GC 只能回收不可达对象。
如果对象仍然被引用，即使业务上已经不需要了，GC 也不会回收。

Go 里常见内存泄漏或内存持续上涨场景包括：

第一，goroutine 泄漏。
goroutine 卡在 channel、select、网络读写、锁等待上，永远不退出。goroutine 栈和引用的对象都无法释放。

第二，slice 截取大数组导致内存滞留。
只保留小 slice，但它仍然引用大数组，导致大数组无法回收。

第三，map 只增不删。
全局 map、缓存 map、连接 map 中 key 不断增加，没有淘汰策略。

第四，time.Ticker 没有 Stop。
Ticker 持续运行，相关资源无法释放。

第五，context 没有 cancel。
WithTimeout/WithCancel 创建后不调用 cancel，可能导致资源释放延迟。

第六，sync.Pool 使用不当。
Put 回去的对象保留大 buffer 或用户对象引用，导致内存占用高。

第七，日志、指标、trace 标签维度爆炸。
例如用 userID、orderID 作为 metric label，导致内部 map 无限增长。

面试里可以这样回答：

Go 有 GC 但仍然会内存泄漏。只要对象仍然可达，GC 就不会回收。常见原因包括 goroutine 泄漏、全局 map 无淘汰、slice 引用大数组、Ticker 未 Stop、context 未 cancel、缓存无上限等。

---

## 【代码块 9】slice 截取导致大数组滞留

```go
import "fmt"

func badSubSlice() []byte {
    big := make([]byte, 1024*1024*100) // 100MB
    return big[:10]
}

small := badSubSlice()

fmt.Println(len(small))
```

---

## 【文本块 31】代码解释

small 只有 10 个字节长度，但它引用的底层数组是 100MB。

只要 small 还活着，这个 100MB 数组就可能无法被 GC 回收。

正确做法是 copy 一份小切片。

---

## 【代码块 10】copy 避免大数组滞留

```go
import "fmt"

func goodSubSlice() []byte {
    big := make([]byte, 1024*1024*100)
    part := big[:10]

    small := make([]byte, len(part))
    copy(small, part)

    return small
}

small := goodSubSlice()

fmt.Println(len(small))
```

---

## 【代码块 11】Ticker 未 Stop 的风险

```go
import (
    "fmt"
    "time"
)

func startTicker() {
    ticker := time.NewTicker(time.Second)

    go func() {
        for t := range ticker.C {
            fmt.Println(t)
        }
    }()

    // 如果没有 ticker.Stop()，这个 ticker 会一直运行
}

startTicker()
time.Sleep(3 * time.Second)
```

---

## 【文本块 32】Ticker 正确写法

如果 ticker 有明确生命周期，应该 Stop。

```go
ticker := time.NewTicker(time.Second)
defer ticker.Stop()
```

如果 goroutine 里使用 ticker，通常还要配合 context 控制退出。

---

## 【代码块 12】用 context 控制 ticker 退出

```go
import (
    "context"
    "fmt"
    "time"
)

func runTicker(ctx context.Context) {
    ticker := time.NewTicker(500 * time.Millisecond)
    defer ticker.Stop()

    for {
        select {
        case <-ctx.Done():
            fmt.Println("ticker stopped")
            return
        case t := <-ticker.C:
            fmt.Println("tick:", t.UnixMilli())
        }
    }
}

ctx, cancel := context.WithCancel(context.Background())

go runTicker(ctx)

time.Sleep(1200 * time.Millisecond)
cancel()
time.Sleep(200 * time.Millisecond)
```

---

# 十四、goroutine 泄漏

## 【文本块 33】Q：什么是 goroutine 泄漏？

goroutine 泄漏是指 goroutine 启动后，因为阻塞或退出条件缺失，永远无法结束。

它虽然不是传统意义上的内存泄漏，但 goroutine 本身占用栈空间，并且可能持有对象引用，导致这些对象也无法被 GC 回收。

常见 goroutine 泄漏场景：

1. channel 读阻塞，永远没人写
2. channel 写阻塞，永远没人读
3. select 没有退出分支
4. 网络请求没有超时
5. 后台 for 循环没有 context 控制
6. worker 等待任务但任务队列永不关闭
7. producer/consumer 一方退出，另一方阻塞

面试里可以这样回答：

goroutine 泄漏本质是协程生命周期没有被正确管理。Go 里启动 goroutine 很便宜，但不是免费的。每个 goroutine 都应该知道什么时候退出，通常通过 context、关闭 channel、超时控制或 WaitGroup 来管理。

---

## 【代码块 13】goroutine 泄漏示例

```go
import (
    "fmt"
    "time"
)

func leak() {
    ch := make(chan int)

    go func() {
        v := <-ch
        fmt.Println(v)
    }()
}

leak()

time.Sleep(time.Second)

fmt.Println("goroutine may be leaked")
```

---

## 【文本块 34】代码解释

这个 goroutine 一直等待从 ch 读取数据，但没有任何地方会写入 ch，也没有关闭 ch。

所以它永远阻塞，形成 goroutine 泄漏。

---

## 【代码块 14】用 context 避免 goroutine 泄漏

```go
import (
    "context"
    "fmt"
    "time"
)

func noLeak(ctx context.Context) {
    ch := make(chan int)

    go func() {
        select {
        case v := <-ch:
            fmt.Println(v)
        case <-ctx.Done():
            fmt.Println("goroutine exit:", ctx.Err())
            return
        }
    }()
}

ctx, cancel := context.WithTimeout(context.Background(), 500*time.Millisecond)
defer cancel()

noLeak(ctx)

time.Sleep(time.Second)
```

---

# 十五、pprof 总览

## 【文本块 35】Q：pprof 是什么？

pprof 是 Go 官方提供的性能分析工具。

它可以帮助我们分析：

* CPU 消耗在哪里
* 内存分配在哪里
* 哪些 goroutine 卡住了
* 哪些锁竞争严重
* 哪些操作阻塞严重
* heap 上哪些对象占用最多
* 程序调用链耗时和分配情况

常见 profile 类型：

1. CPU profile
   看 CPU 时间主要消耗在哪些函数。

2. heap profile
   看内存分配和存活对象情况。

3. goroutine profile
   看当前 goroutine 数量、栈、阻塞位置。

4. mutex profile
   看锁竞争情况。

5. block profile
   看 channel、select、mutex 等阻塞情况。

6. allocs profile
   看历史分配情况。

面试里可以这样回答：

pprof 是 Go 性能分析的核心工具。CPU 高看 CPU profile，内存高看 heap profile，goroutine 数量异常看 goroutine profile，锁竞争看 mutex profile，阻塞严重看 block profile。调优时应该先根据现象选择 profile，而不是盲目猜。

---

# 十六、HTTP 服务开启 pprof

## 【文本块 36】Q：Go 服务怎么开启 pprof？

最常见方式是引入：

```go
import _ "net/http/pprof"
```

然后启动一个 HTTP 服务：

```go
go http.ListenAndServe(":6060", nil)
```

这样就可以访问：

```text
/debug/pprof/
```

常见地址：

```text
/debug/pprof/profile
/debug/pprof/heap
/debug/pprof/goroutine
/debug/pprof/mutex
/debug/pprof/block
```

生产环境注意：

pprof 暴露的信息非常敏感，不能直接暴露到公网。
应该只绑定 localhost、内网地址，或者加鉴权、通过 VPN/堡垒机访问。

---

## 【代码块 15】开启 pprof HTTP 服务

```go
// 保存为 main.go 后运行：go run main.go
// 浏览器访问：http://localhost:6060/debug/pprof/

package main

import (
    "fmt"
    "net/http"
    _ "net/http/pprof"
    "time"
)

func main() {
    go func() {
        fmt.Println(http.ListenAndServe("localhost:6060", nil))
    }()

    for {
        time.Sleep(time.Second)
    }
}
```

---

## 【文本块 37】常用 pprof 命令

CPU profile：

```bash
go tool pprof http://localhost:6060/debug/pprof/profile?seconds=30
```

Heap profile：

```bash
go tool pprof http://localhost:6060/debug/pprof/heap
```

Goroutine profile：

```bash
go tool pprof http://localhost:6060/debug/pprof/goroutine
```

查看 Web 图：

```bash
go tool pprof -http=:8080 http://localhost:6060/debug/pprof/heap
```

进入交互模式后常用命令：

```text
top
top -cum
list functionName
web
peek functionName
```

top 看当前函数自身占用。
top -cum 看累计调用链占用。
list 可以看某个函数源码级消耗。
web 可以看调用图。

---

# 十七、CPU Profile

## 【文本块 38】Q：CPU 高怎么用 pprof 排查？

CPU 高时，第一步不是马上改代码，而是抓 CPU profile。

常用命令：

```bash
go tool pprof http://localhost:6060/debug/pprof/profile?seconds=30
```

进入后先看：

```text
top
top -cum
```

重点关注：

* flat 高的函数：函数自身消耗 CPU 多
* cum 高的函数：函数及其调用链整体消耗高
* 是否有大量 JSON 编解码
* 是否有正则、加密、压缩
* 是否有复杂排序或循环
* 是否有锁自旋或忙等
* 是否有日志格式化开销
* 是否有反射开销

面试里可以这样回答：

CPU 高时，我会抓 CPU profile，看 top 和 top -cum。flat 高说明函数自身耗 CPU，cum 高说明调用链整体耗 CPU。然后结合源码 list 定位热点，是 JSON、正则、排序、锁竞争还是业务循环，再针对性优化。

---

## 【代码块 16】CPU 热点示例

```go
// 保存为 cpu_hot.go 后运行，并用 pprof 抓 CPU
package main

import (
    "fmt"
    "net/http"
    _ "net/http/pprof"
)

func busyWork() {
    x := 0
    for i := 0; i < 1_000_000; i++ {
        x += i
    }
    _ = x
}

func main() {
    go http.ListenAndServe("localhost:6060", nil)

    for {
        busyWork()
        fmt.Print("")
    }
}
```

---

## 【文本块 39】CPU 优化常见方向

CPU profile 定位后，常见优化方向包括：

1. 减少重复计算
2. 使用更合适的数据结构
3. 减少 JSON/反射开销
4. 避免在热点路径使用 fmt.Sprintf
5. 优化正则表达式或避免重复编译正则
6. 减少排序数据量
7. 批量处理代替逐条处理
8. 避免忙等循环
9. 降低锁竞争
10. 使用缓存

不要在没有 profile 的情况下凭感觉优化。
Go 调优最重要的是“证据驱动”。

---

# 十八、Heap Profile

## 【文本块 40】Q：内存高怎么用 pprof 排查？

内存高时，优先看 heap profile。

命令：

```bash
go tool pprof http://localhost:6060/debug/pprof/heap
```

heap profile 有两个常用视角：

1. inuse_space
   当前还活着的对象占用多少内存。适合排查内存泄漏、常驻内存高。

2. alloc_space
   历史累计分配了多少内存。适合排查分配频繁、GC 压力大。

常用命令：

```text
top
top -cum
```

也可以显式指定：

```bash
go tool pprof -inuse_space http://localhost:6060/debug/pprof/heap
go tool pprof -alloc_space http://localhost:6060/debug/pprof/heap
```

面试里可以这样回答：

内存高我会先看 heap profile。如果关注当前谁占内存，用 inuse_space；如果关注谁分配得多、导致 GC 压力大，用 alloc_space。然后结合 top、top -cum 和 list 定位具体分配点。

---

## 【文本块 41】inuse_space 和 alloc_space 怎么区分？

inuse_space 看的是当前仍然存活的对象。
它更适合排查“内存为什么降不下来”。

例如：

* 全局 map 持有大量对象
* 缓存没有淘汰
* goroutine 泄漏持有引用
* slice 截取导致大数组滞留

alloc_space 看的是历史累计分配量。
它更适合排查“为什么 GC 很频繁”。

例如：

* 每个请求都创建大量临时对象
* JSON 编解码分配很多
* 字符串拼接产生大量临时对象
* 日志格式化频繁分配
* 反射导致临时对象多

一句话：

> inuse_space 看存量，alloc_space 看流量。

---

## 【代码块 17】内存分配热点示例

```go
// 保存为 heap_hot.go 后运行，并用 pprof 看 heap
package main

import (
    "net/http"
    _ "net/http/pprof"
    "time"
)

var data [][]byte

func allocate() {
    for i := 0; i < 1000; i++ {
        b := make([]byte, 1024*100)
        data = append(data, b)
    }
}

func main() {
    go http.ListenAndServe("localhost:6060", nil)

    for {
        allocate()
        time.Sleep(time.Second)
    }
}
```

---

## 【文本块 42】代码解释

这个程序会不断往全局变量 data 里追加 byte slice。

因为 data 是全局变量，里面引用的对象一直可达，所以 GC 不会回收这些 byte slice。

heap profile 的 inuse_space 会看到内存持续增长。

这就是典型的“全局容器只增不删”导致的内存泄漏。

---

# 十九、Goroutine Profile

## 【文本块 43】Q：goroutine 数量暴涨怎么排查？

goroutine 数量异常时，看 goroutine profile。

命令：

```bash
go tool pprof http://localhost:6060/debug/pprof/goroutine
```

或者直接访问：

```text
/debug/pprof/goroutine?debug=2
```

debug=2 会输出所有 goroutine 的栈信息。

重点看：

* 大量 goroutine 是否卡在同一行
* 是否卡在 channel receive
* 是否卡在 channel send
* 是否卡在 select
* 是否卡在 mutex Lock
* 是否卡在 net/http 请求
* 是否卡在 database/sql
* 是否卡在 time.Sleep 或 ticker
* 是否有 goroutine 创建后没有退出条件

面试里可以这样回答：

goroutine 暴涨时，我会看 goroutine profile，尤其是 debug=2 的栈。重点找大量 goroutine 卡在同一个位置，比如 channel 读写、锁等待、网络请求、数据库查询。找到阻塞点后，再看是否缺少 timeout、context cancel、channel close 或 worker 退出机制。

---

## 【代码块 18】goroutine 泄漏 profile 示例

```go
// 保存为 goroutine_leak.go 后运行
package main

import (
    "net/http"
    _ "net/http/pprof"
    "time"
)

func leak() {
    ch := make(chan struct{})

    go func() {
        <-ch
    }()
}

func main() {
    go http.ListenAndServe("localhost:6060", nil)

    for {
        leak()
        time.Sleep(10 * time.Millisecond)
    }
}
```

---

## 【文本块 44】如何修复 goroutine 泄漏？

修复思路一般有：

1. 给 goroutine 明确退出条件
2. 使用 context 控制生命周期
3. 关闭 channel 通知退出
4. 网络/数据库/RPC 调用设置 timeout
5. producer/consumer 两端都处理退出
6. worker pool 关闭时 close job channel
7. 使用 errgroup 管理一组 goroutine
8. 在 goroutine 入口 recover，避免 panic 导致状态异常

核心原则：

> 每个 goroutine 启动时，都要想清楚它什么时候退出。

---

# 二十、Mutex Profile

## 【文本块 45】Q：锁竞争严重怎么排查？

如果怀疑锁竞争，可以开启 mutex profile。

代码中需要设置采样比例：

```go
runtime.SetMutexProfileFraction(1)
```

然后通过 pprof 查看：

```bash
go tool pprof http://localhost:6060/debug/pprof/mutex
```

mutex profile 主要用于分析 goroutine 在锁上等待的时间。

常见问题：

* 大锁保护了过多逻辑
* 临界区里做了 IO
* 临界区里做了复杂计算
* 全局 map 用一把锁
* 热点 key 竞争严重
* 日志或指标内部锁竞争
* RWMutex 写锁饥饿或读写比例不适合

优化方向：

1. 缩小锁粒度
2. 减少临界区耗时
3. 临界区内不要做 IO
4. 分片锁
5. 使用 atomic
6. 使用 channel 串行化
7. 使用 sync.Map 适配读多写少
8. 重新设计数据结构

---

## 【代码块 19】开启 mutex profile

```go
// 在 main 函数里开启
import "runtime"

runtime.SetMutexProfileFraction(1)
```

---

## 【文本块 46】Mutex 调优怎么回答更像工程实践？

不要一上来就说“换 sync.Map”。

更稳的回答顺序是：

第一，先用 mutex profile 确认是否真的是锁竞争。
第二，看是哪把锁、哪个调用链等待时间高。
第三，检查临界区是否过大，是否包含 IO、RPC、DB、日志等慢操作。
第四，再考虑缩小锁范围、拆分锁、分片、atomic 或无锁结构。
第五，用 benchmark 或压测验证优化效果。

一句话：

> 锁优化不是简单换工具，而是先定位竞争点，再缩短持锁时间、降低共享状态、拆分热点。

---

# 二十一、Block Profile

## 【文本块 47】Q：block profile 看什么？

block profile 用于分析 goroutine 阻塞情况。

它关注的不只是 mutex，还包括：

* channel send 阻塞
* channel receive 阻塞
* select 阻塞
* mutex 阻塞
* cond 阻塞
* 网络或同步等待相关阻塞

开启方式：

```go
runtime.SetBlockProfileRate(1)
```

查看：

```bash
go tool pprof http://localhost:6060/debug/pprof/block
```

block profile 适合排查：

* 程序吞吐下降但 CPU 不高
* goroutine 很多但都在等
* channel 使用不当
* worker pool 堵塞
* 下游慢导致上游阻塞
* 锁竞争或条件等待明显

面试里可以这样回答：

CPU 高看 CPU profile，锁竞争看 mutex profile；如果 CPU 不高但请求卡住、goroutine 很多，就看 block profile 和 goroutine profile，定位是 channel 阻塞、锁等待还是下游 IO 等待。

---

## 【代码块 20】开启 block profile

```go
// 在 main 函数里开启
import "runtime"

runtime.SetBlockProfileRate(1)
```

---

# 二十二、runtime.MemStats

## 【文本块 48】Q：除了 pprof，还能怎么看 Go 内存状态？

可以使用 runtime.ReadMemStats 读取内存统计。

常见字段：

* Alloc：当前堆上已分配且仍在使用的字节数
* TotalAlloc：程序启动以来累计分配的字节数
* Sys：从操作系统获得的总内存
* HeapAlloc：当前堆分配
* HeapSys：从 OS 申请的堆内存
* HeapIdle：空闲 span
* HeapReleased：已经归还给 OS 的内存
* NumGC：GC 次数
* PauseTotalNs：GC 总暂停时间
* LastGC：上次 GC 时间

MemStats 适合做调试和监控，但线上更常用 Prometheus runtime metrics 或 OpenTelemetry 指标。

---

## 【代码块 21】读取 runtime.MemStats

```go
import (
    "fmt"
    "runtime"
)

var m runtime.MemStats

runtime.ReadMemStats(&m)

fmt.Println("Alloc MB:", m.Alloc/1024/1024)
fmt.Println("TotalAlloc MB:", m.TotalAlloc/1024/1024)
fmt.Println("Sys MB:", m.Sys/1024/1024)
fmt.Println("NumGC:", m.NumGC)
```

---

## 【文本块 49】Alloc、TotalAlloc、Sys 怎么理解？

Alloc 是当前还在用的堆内存。
如果 Alloc 持续上涨且 GC 后不下降，要怀疑内存泄漏或缓存无上限。

TotalAlloc 是累计分配。
它只增不减，适合看分配总量，不能用来判断当前内存是否泄漏。

Sys 是 runtime 从操作系统申请的总内存。
它可能比 Alloc 大很多，因为 runtime 申请过的内存不一定立刻还给 OS，可能保留着供后续复用。

所以看到 RSS 或 Sys 高，不一定说明业务对象都还活着。要结合 HeapAlloc、HeapIdle、HeapReleased、pprof heap 一起看。

---

# 二十三、Go 调优基本思路

## 【文本块 50】Q：你做过 Go 性能调优吗？一般怎么做？

Go 调优我一般按四步走：

第一步，看现象。

常见现象包括：

* CPU 使用率高
* 内存持续上涨
* GC 频繁
* 接口 RT 抖动
* goroutine 数量暴涨
* 锁等待严重
* 请求吞吐下降
* 下游调用超时
* 容器频繁 OOMKilled

第二步，收集证据。

对应工具：

* CPU 高：CPU profile
* 内存高：heap profile
* goroutine 异常：goroutine profile
* 锁竞争：mutex profile
* 阻塞严重：block profile
* GC 问题：gctrace、runtime metrics、MemStats
* 延迟问题：trace、日志、链路追踪

第三步，定位根因。

比如：

* 是 JSON 编解码太重？
* 是 map 锁竞争？
* 是 goroutine 泄漏？
* 是缓存无限增长？
* 是 slice 截取导致大数组滞留？
* 是 DB/RPC 没有 timeout？
* 是每个请求分配太多对象？
* 是 GC assist 影响延迟？

第四步，验证优化。

优化后要通过：

* benchmark
* 压测
* pprof 对比
* 线上灰度指标
* p99 延迟
* GC 次数和暂停
* 内存水位

来确认真的改善了。

面试里可以这样回答：

Go 调优不是先改代码，而是先根据现象选工具。CPU 高抓 CPU profile，内存高抓 heap，goroutine 涨看 goroutine profile，锁竞争看 mutex，阻塞看 block。定位具体函数和调用链后，再做数据结构、并发模型、对象分配、缓存策略等优化，并通过压测和 pprof 对比验证。

---

# 二十四、常见调优场景一：CPU 高

## 【文本块 51】Q：线上 Go 服务 CPU 飙高怎么办？

我会按下面顺序排查：

第一，看是否是业务流量上涨。
先看 QPS、请求类型、是否有突发流量。

第二，看 CPU profile。
抓 30 秒 CPU profile，看 top 和 top -cum。

第三，看热点函数。
常见 CPU 热点包括：

* JSON marshal/unmarshal
* fmt.Sprintf
* 正则匹配
* 加密解密
* 压缩解压
* 大量排序
* 复杂循环
* 反射
* map 热点访问
* 日志格式化
* 自旋或忙等

第四，看是否 GC CPU 高。
如果 CPU profile 里 runtime.gcBgMarkWorker、runtime.mallocgc 等占比高，说明 GC 和分配压力较大。

第五，针对性优化。
比如减少分配、缓存结果、批量处理、优化算法、减少反射、减少日志、限制并发。

面试里可以这样回答：

CPU 高不能盲目猜，先抓 CPU profile。看 flat 和 cum 找热点。如果热点在业务函数，就优化算法或减少重复计算；如果热点在 JSON/反射，就考虑更高效编码或减少动态处理；如果热点在 runtime.mallocgc 或 GC worker，就说明分配和 GC 压力大，要看 heap 和 benchmem。

---

# 二十五、常见调优场景二：内存持续上涨

## 【文本块 52】Q：Go 服务内存一直涨怎么办？

我会先区分两件事：

第一，是堆对象真的一直涨？
看 HeapAlloc、heap inuse_space。

第二，是 runtime 向 OS 申请的内存没有立即归还？
看 Sys、HeapIdle、HeapReleased、RSS。

如果 HeapAlloc 持续上涨，GC 后也不下降，要怀疑内存泄漏。

常见原因：

* 全局 map/cache 没有淘汰
* goroutine 泄漏持有引用
* slice 小切片引用大数组
* 连接或请求对象没有释放
* Ticker/Timer 泄漏
* 指标 label 维度爆炸
* sync.Pool 持有大对象
* 队列堆积

排查工具：

* heap profile 的 inuse_space
* goroutine profile
* runtime.MemStats
* 业务缓存大小指标
* 队列长度指标
* pprof web 调用图

面试里可以这样回答：

内存持续上涨先看 heap profile 的 inuse_space。如果是某个全局 map、cache 或 slice 持有对象，就检查淘汰策略和引用链。如果 goroutine 数量也涨，要看 goroutine profile，因为泄漏的 goroutine 可能持有大量对象。如果 HeapAlloc 不高但 RSS 高，要进一步看 HeapIdle/HeapReleased，可能是 runtime 尚未归还 OS。

---

# 二十六、常见调优场景三：GC 频繁

## 【文本块 53】Q：GC 很频繁怎么办？

GC 频繁通常说明对象分配速度快，或者堆目标太小。

排查思路：

第一，看分配速率。
通过 runtime metrics、pprof alloc_space、benchmark 的 B/op 和 allocs/op。

第二，看主要分配点。
heap profile 用 alloc_space 视角，找累计分配最多的函数。

第三，看对象是否短生命周期。
如果大量对象很快死亡，说明可以通过复用、减少临时对象、批量处理优化。

第四，看 GOGC 是否过低。
如果内存充足但 GC 过于频繁，可以适当调高 GOGC。

第五，看是否有大对象频繁分配。
比如每个请求都 make 大 buffer、读全量文件、构造大 JSON。

优化方向：

* 减少临时对象
* 使用 strings.Builder / bytes.Buffer
* 复用 buffer
* 使用 sync.Pool
* 避免循环中 fmt.Sprintf
* 避免频繁 []byte/string 转换
* 批量处理
* 调整 GOGC
* 优化数据结构

面试里可以这样回答：

GC 频繁的核心通常是分配太快。排查时用 heap profile 的 alloc_space 找累计分配热点，用 benchmark -benchmem 看 allocs/op。优化上先减少对象分配和临时对象，再考虑 sync.Pool 或调 GOGC。调 GOGC 是手段，不应该掩盖对象分配异常这个根因。

---

# 二十七、常见调优场景四：接口 RT 抖动

## 【文本块 54】Q：Go 服务接口延迟抖动怎么排查？

接口 RT 抖动可能不是单一原因。

我会从几个方向看：

第一，GC 是否影响延迟。
看 GC 次数、暂停时间、GC assist、heap 增长。

第二，锁竞争。
看 mutex profile，是否大量 goroutine 等同一把锁。

第三，阻塞。
看 block profile 和 goroutine profile，是否卡在 channel、select、DB、RPC。

第四，下游依赖。
看数据库、Redis、MQ、RPC 的耗时和超时。

第五，调度压力。
goroutine 数量是否过多，是否有大量 runnable goroutine。

第六，CPU 饱和。
CPU 接近打满时，调度排队会导致延迟抖动。

第七，日志和指标。
同步日志、指标高基数、trace 采样过高都可能导致抖动。

面试里可以这样回答：

RT 抖动要结合链路追踪和 pprof。先看是不是下游慢，再看 GC、CPU、锁竞争、阻塞和 goroutine 数量。如果 GC assist 明显，说明分配压力影响请求；如果 mutex profile 高，说明锁竞争；如果 block/goroutine 显示大量请求卡在 channel 或 RPC，就要处理背压和超时。

---

# 二十八、gctrace

## 【文本块 55】Q：怎么查看 Go GC 日志？

可以通过环境变量开启 gctrace：

```bash
GODEBUG=gctrace=1 ./app
```

运行后，程序每次 GC 会打印一行日志。

日志里会包含 GC 次数、时间、STW、堆大小、CPU 使用等信息。

面试里不要求逐字段背下来，但要知道 gctrace 可以帮助判断：

* GC 是否频繁
* 每次 GC 前后堆大小
* STW 时间
* GC CPU 消耗
* 堆目标变化

常见用途：

* 本地压测时观察 GC
* 对比优化前后 GC 次数
* 判断是否分配过快
* 验证 GOGC 调整效果

---

## 【代码块 22】开启 gctrace

```bash
GODEBUG=gctrace=1 go run main.go
```

---

## 【文本块 56】怎么解读 gctrace？

大致关注：

1. GC 发生频率
   如果日志刷得很快，说明 GC 频繁。

2. 堆大小变化
   看 GC 前后堆是否能明显下降。
   如果 GC 后仍然很高，可能存活对象多或泄漏。

3. STW 时间
   如果 STW 明显变长，可能影响延迟。

4. GC CPU 占比
   如果 GC CPU 高，说明 GC 对业务吞吐有影响。

面试里可以说：

> 我不会只靠 gctrace 下结论，而是把它和 heap profile、alloc_space、runtime metrics 一起看。gctrace 适合看 GC 频率和整体趋势，pprof 适合定位具体分配点。

---

# 二十九、Go Trace

## 【文本块 57】Q：go tool trace 是什么？

pprof 更擅长看 CPU、内存、goroutine 栈和锁等待。
go tool trace 更擅长看时间线。

trace 可以看到：

* goroutine 什么时候创建
* goroutine 什么时候运行
* goroutine 什么时候阻塞
* 网络阻塞
* syscall 阻塞
* GC 事件
* STW
* 调度延迟
* processor 利用情况
* goroutine runnable 但没被调度的时间

如果问题是“为什么请求偶尔卡一下”，trace 有时比 pprof 更直观。

常见使用方式：

```bash
go test -trace trace.out
go tool trace trace.out
```

或者在程序中用 runtime/trace 采集。

面试里可以这样回答：

pprof 适合看资源消耗聚合结果，trace 适合看运行时间线和调度行为。如果怀疑是调度延迟、GC 暂停、网络阻塞、goroutine 阻塞导致的延迟抖动，可以用 go tool trace 进一步分析。

---

# 三十、编译器优化：内联和边界检查消除

## 【文本块 58】Q：Go 编译器会做哪些常见优化？

Go 虽然没有 JVM JIT，但编译器会做很多静态优化。

常见优化包括：

1. 函数内联
   小函数会被直接展开到调用处，减少函数调用开销。

2. 逃逸分析
   决定变量分配在栈还是堆。

3. 边界检查消除
   对 slice/array 下标访问，编译器在能证明安全时会消除 bounds check。

4. 死代码消除
   不会执行或无用代码会被删掉。

5. 常量折叠
   编译期能算出来的表达式直接算好。

6. 去虚化
   某些接口调用在编译器能确定具体类型时，可以优化成直接调用。

面试里可以这样回答：

Go 没有传统 JIT，但编译器会做 AOT 优化，比如内联、逃逸分析、边界检查消除、死代码消除等。因此 Go 启动快、性能稳定，但运行时不会像 JVM 那样根据热点动态做大规模 JIT 优化。

---

## 【文本块 59】Q：什么是边界检查消除？

Go 对 slice 和 array 下标访问会做边界检查，防止越界 panic。

例如：

```go
x := s[i]
```

runtime 需要确保 i 在合法范围内。

但如果编译器能证明 i 一定合法，就可以消除这次边界检查，减少开销。

例如：

```go
for i := 0; i < len(s); i++ {
    _ = s[i]
}
```

这种常见循环里，编译器通常能证明 i 不会越界。

如果代码写得复杂，编译器证明不了，就可能保留边界检查。

面试里不需要深入汇编，但知道这个概念即可。

---

# 三十一、本章高频面试题速记

## 【文本块 60】内存分配速记

Go 变量分配在栈还是堆，由逃逸分析决定，不由 new 或取地址简单决定。

返回局部变量指针是安全的，因为会逃逸到堆上。

new 不一定堆分配，make 也不等于堆分配。

栈分配快，函数返回自动释放，不需要 GC。

堆分配更灵活，但会增加 GC 压力。

小对象通常从 P 的 mcache 分配，mcache 不够找 mcentral，再找 mheap，最后向 OS 申请。

mspan 是连续内存页，按 size class 切成固定大小槽位。

大量小对象虽然分配快，但会增加 allocs/op 和 GC 扫描压力。

sync.Pool 适合复用临时对象，不适合做连接池。

---

## 【文本块 61】逃逸分析速记

常见逃逸场景：

* 返回局部变量指针
* 闭包捕获变量
* 存入全局变量
* 存入堆对象
* 传入 interface 导致不确定
* 对象过大
* 生命周期超过当前函数

查看逃逸分析：

```bash
go build -gcflags="-m" main.go
```

减少逃逸：

* 小对象尽量传值
* 减少不必要指针返回
* 避免热点路径 interface{} 和反射
* 减少闭包捕获
* 复用 buffer
* 用 benchmark -benchmem 验证

---

## 【文本块 62】GC 速记

Go GC 是并发三色标记清扫。

三色：

* 白色：未标记
* 灰色：已标记但引用未扫描完
* 黑色：已标记且引用扫描完

并发标记会有漏标风险，需要写屏障。

Go GC 有 STW，但主要 STW 阶段很短。

GOGC 控制 GC 触发的堆增长比例。

GOGC 越大，GC 越少，内存越高。
GOGC 越小，GC 越频繁，内存越低。

GC assist 是分配太快时，让业务 goroutine 帮忙做 GC 标记。

Go 有 GC 也会内存泄漏，因为可达但无用的对象不会被回收。

---

## 【文本块 63】pprof 速记

CPU 高：看 CPU profile。

```bash
go tool pprof http://localhost:6060/debug/pprof/profile?seconds=30
```

内存高：看 heap profile。

```bash
go tool pprof http://localhost:6060/debug/pprof/heap
```

goroutine 泄漏：看 goroutine profile。

```bash
go tool pprof http://localhost:6060/debug/pprof/goroutine
```

锁竞争：看 mutex profile。

```go
runtime.SetMutexProfileFraction(1)
```

阻塞严重：看 block profile。

```go
runtime.SetBlockProfileRate(1)
```

pprof 常用命令：

```text
top
top -cum
list
web
peek
```

heap 两个视角：

* inuse_space：当前谁占内存
* alloc_space：历史谁分配多

一句话：

> inuse 看存量，alloc 看流量。

---

## 【文本块 64】Go 调优速记

调优顺序：

1. 看现象
2. 抓 profile
3. 找热点
4. 改代码
5. 压测验证
6. 线上灰度观察

CPU 高：

* CPU profile
* 看 flat/cum
* 找业务热点或 runtime 热点

内存高：

* heap inuse_space
* 看谁还活着
* 查缓存、map、goroutine、slice

GC 频繁：

* heap alloc_space
* benchmem
* 减少分配
* 复用对象
* 必要时调 GOGC

goroutine 多：

* goroutine profile debug=2
* 找阻塞栈
* context/timeout/close channel

锁竞争：

* mutex profile
* 缩小临界区
* 分片锁
* atomic
* sync.Map
* 重新设计数据结构

RT 抖动：

* 链路追踪
* GC 指标
* CPU
* mutex/block profile
* goroutine profile
* 下游耗时

---

# 三十二、本章最终面试回答模板

## 【文本块 65】综合回答模板

如果面试官让我整体讲 Go Runtime，我会这样回答：

Go runtime 是 Go 程序运行时的基础设施，主要负责 goroutine 调度、内存分配、垃圾回收、栈管理、网络轮询，以及 map、channel、defer、panic 等语言特性的底层支持。Go 虽然是编译型语言，没有 JVM 那种字节码虚拟机，但它有一个非常重要的 runtime，并且会被链接进最终二进制中。

在内存分配上，Go 变量最终分配在栈还是堆，主要由逃逸分析决定，而不是由 new 或取地址简单决定。如果变量生命周期只在当前函数内，通常可以放在栈上；如果返回局部变量指针、被闭包捕获、存入全局变量或编译器无法证明安全，就可能逃逸到堆上。栈分配速度快、无需 GC；堆分配更灵活，但会增加 GC 压力。

Go 的内存分配器采用分级缓存设计。小对象优先从当前 P 的 mcache 分配，mcache 不够再找 mcentral，mcentral 不够找 mheap，最后才向操作系统申请。通过 mspan 和 size class，Go 可以高效管理大量小对象。对于非常小且不含指针的对象，还有 tiny allocator 优化。但大量小对象进入堆后仍然会增加 GC 扫描压力，所以热点路径要关注 B/op 和 allocs/op。

Go GC 是并发三色标记清扫 GC。它从 GC Roots 出发标记可达对象，通过白、灰、黑三色描述扫描状态。并发标记期间，用户 goroutine 也在修改引用关系，所以需要写屏障防止漏标。Go GC 不是完全没有 STW，而是把主要 STW 控制得很短，大部分标记和清扫工作与用户 goroutine 并发执行。GOGC 控制 GC 触发的堆增长比例，调大可以减少 GC 频率但增加内存，调小可以降低内存但增加 GC CPU。

Go 有 GC 仍然会发生内存泄漏，因为 GC 只能回收不可达对象。常见问题包括 goroutine 泄漏、全局 map/cache 无淘汰、slice 小切片引用大数组、Ticker 未 Stop、context 未 cancel、指标维度爆炸等。排查时要结合 heap profile、goroutine profile 和 runtime 指标。

Go 调优一般不是凭感觉改代码，而是根据现象选择工具。CPU 高看 CPU profile，内存高看 heap profile，goroutine 暴涨看 goroutine profile，锁竞争看 mutex profile，阻塞严重看 block profile。heap profile 里 inuse_space 看当前存活对象，alloc_space 看历史累计分配。定位到具体函数后，再从算法、数据结构、对象复用、减少分配、降低锁竞争、控制 goroutine 生命周期等方向优化，并通过 benchmark、压测和线上指标验证。

一句话总结：Go runtime 面试要抓住三条主线，第一是逃逸分析决定栈堆分配，第二是 mcache/mcentral/mheap 支撑高效内存分配，第三是并发三色标记 GC 配合 pprof 工具链完成性能定位和调优。
